# Tự Động Huấn Luyện & So Sánh Nhiều Cấu Hình (Chạy Được Cả Local & Kaggle)
Notebook này được tối ưu sẵn để **chạy trực tiếp trên Kaggle GPU (T4 / P100)** hoặc trên máy cá nhân:
1. **Tự động nhận diện môi trường:** Tự động tìm đường dẫn dataset GoPro trong `/kaggle/input` hoặc thư mục local.
2. **Danh sách cấu hình (`CONFIG_LIST`):** Tự động huấn luyện lần lượt từng cấu hình qua vòng lặp `for`.
3. **Lưu trữ riêng biệt:** Mỗi cấu hình có file checkpoint `.pth` riêng.
4. **Bảng xếp hạng (Leaderboard):** Tự động xuất bảng so sánh PSNR, SSIM, Loss của từng cấu hình.
5. **Biểu đồ đa đường:** Vẽ các đường cong hội tụ đè lên nhau để so sánh trực quan.
6. **Visual Demo:** Render ảnh so sánh 3 cột: [Ảnh Mờ] vs [Ảnh AI Phục Hồi] vs [Ảnh Gốc Nét].

In [ ]:
import sys
import os
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

# 1. Tự động nhận diện thư mục chứa models và utils dù ở Local hay Kaggle
candidate_paths = [
    '.', '..', 
    '/kaggle/working', 
    '/kaggle/working/Image-processing-project-integrating-MoE-and-Q-Former'
]
for p in candidate_paths:
    if os.path.exists(os.path.join(p, 'models')) and os.path.exists(os.path.join(p, 'utils')):
        abs_p = os.path.abspath(p)
        if abs_p not in sys.path:
            sys.path.insert(0, abs_p)
        print(f"--> Đã nhận diện mã nguồn dự án tại: {abs_p}")
        break

from models.moe_qformer_net import MoEQFormerDeblurNet
from utils.dataset import GoProDeblurDataset
from utils.metrics import CharbonnierLoss, calc_psnr, calc_ssim, count_parameters

# 2. Kiểm tra phần cứng GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" Thiết bị đang sử dụng: {device}")
if torch.cuda.is_available():
    print(f" GPU Model: {torch.cuda.get_device_name(0)}")
    print(f" Bộ nhớ VRAM khả dụng: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

### Bảng Cấu Hình & Tự Động Định Vị Dataset (`CONFIG_LIST`)
Hàm `auto_detect_gopro()` sẽ tự động tìm kiếm thư mục GoPro dù bạn đặt ở Local hay tải từ Kaggle Datasets:

In [ ]:
# Hàm tự động tìm thư mục GoPro train/test
def auto_detect_gopro():
    # Kiểm tra trên Kaggle (/kaggle/input)
    if os.path.exists('/kaggle/input'):
        for root, dirs, _ in os.walk('/kaggle/input'):
            if 'train' in dirs and 'test' in dirs:
                print(f" [Kaggle] Tự động phát hiện GoPro dataset tại: {root}")
                return os.path.join(root, 'train'), os.path.join(root, 'test')
    # Kiểm tra trên máy cá nhân (Local)
    for prefix in ['..', '.']:
        t_train = os.path.join(prefix, 'gopro-data', 'train')
        t_test = os.path.join(prefix, 'gopro-data', 'test')
        if os.path.exists(t_train):
            print(f" [Local] Tự động phát hiện GoPro dataset tại: {os.path.join(prefix, 'gopro-data')}")
            return t_train, t_test
    return '../gopro-data/train', '../gopro-data/test'

gopro_train_dir, gopro_test_dir = auto_detect_gopro()

# Cấu hình chung
COMMON_CONFIG = {
    'train_dir': gopro_train_dir,
    'test_dir': gopro_test_dir,
    'crop_size': 256,               # Kích thước patch cắt ảnh khi train
    'aux_weight': 0.01,             # Trọng số cân bằng tải MoE
    'num_feats': 64,
    'num_blocks': 8,
    'num_queries': 16,
    'num_experts': 3,
    'max_eval_samples': 30          # Số lượng ảnh test dùng để đánh giá nhanh mỗi epoch
}

# DANH SÁCH CÁC CẤU HÌNH CẦN SO SÁNH (Mỗi dict là 1 thí nghiệm)
CONFIG_LIST = [
    {
        'name': 'Exp1_LR1e-4_Freeze', 
        'lr': 1e-4, 
        'batch_size': 8, 
        'epochs': 5, 
        'freeze_backbone': True,
        'description': 'Đóng băng backbone, LR nhỏ 1e-4'
    },
    {
        'name': 'Exp2_LR2e-4_Freeze', 
        'lr': 2e-4, 
        'batch_size': 8, 
        'epochs': 5, 
        'freeze_backbone': True,
        'description': 'Đóng băng backbone, tăng LR lên 2e-4'
    },
    {
        'name': 'Exp3_LR5e-5_Unfreeze', 
        'lr': 5e-5, 
        'batch_size': 8, 
        'epochs': 5, 
        'freeze_backbone': False,
        'description': 'Mở khóa backbone (Full train) với LR nhỏ 5e-5'
    }
]

print(f"\n Tổng cộng có {len(CONFIG_LIST)} cấu hình được lên lịch thử nghiệm:")
for i, cfg in enumerate(CONFIG_LIST):
    print(f"  [{i+1}] {cfg['name']:24s} | LR={cfg['lr']:.0e} | Batch={cfg['batch_size']} | Freeze={str(cfg['freeze_backbone']):5s} | {cfg['description']}")

In [ ]:
# Khởi tạo Dataset dùng chung cho các thí nghiệm
train_ds = GoProDeblurDataset(COMMON_CONFIG['train_dir'], crop_size=COMMON_CONFIG['crop_size'], is_train=True)
test_ds = GoProDeblurDataset(COMMON_CONFIG['test_dir'], is_train=False)

print(f" Đã nạp thành công: {len(train_ds):,} ảnh Train | {len(test_ds):,} ảnh Test")

In [ ]:
# Hàm đánh giá PSNR & SSIM trên tập Test
def evaluate_model(model, loader, max_eval_samples=30):
    model.eval()
    psnr_list, ssim_list = [], []
    count = 0
    
    with torch.no_grad():
        for inputs, targets in loader:
            inputs, targets = inputs.to(device), targets.to(device)
            preds, _ = model(inputs)
            preds = torch.clamp(preds, 0.0, 1.0)
            
            psnr_list.append(calc_psnr(preds, targets))
            ssim_list.append(calc_ssim(preds, targets))
            count += 1
            if max_eval_samples and count >= max_eval_samples:
                break
                
    avg_psnr = sum(psnr_list) / len(psnr_list)
    avg_ssim = sum(ssim_list) / len(ssim_list)
    return avg_psnr, avg_ssim

### Vòng Lặp Huấn Luyện Tự Động Qua Từng Cấu Hình (`for cfg in CONFIG_LIST`)

In [ ]:
# Nơi lưu trữ toàn bộ lịch sử và kết quả của tất cả các cấu hình
EXPERIMENT_RESULTS = {}
criterion = CharbonnierLoss()

for exp_idx, cfg in enumerate(CONFIG_LIST):
    exp_name = cfg['name']
    print("\n" + "="*65)
    print(f" CHẠY THÍ NGHIỆM [{exp_idx + 1}/{len(CONFIG_LIST)}]: {exp_name}")
    print(f"   Mô tả: {cfg['description']}")
    print(f"   LR={cfg['lr']:.0e} | BatchSize={cfg['batch_size']} | Epochs={cfg['epochs']} | FreezeBackbone={cfg['freeze_backbone']}")
    print("="*65)
    
    # 1. DataLoader theo batch_size của cấu hình này
    train_loader = DataLoader(
        train_ds, 
        batch_size=cfg['batch_size'], 
        shuffle=True, 
        num_workers=2, 
        pin_memory=True if torch.cuda.is_available() else False
    )
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False)
    
    # 2. Khởi tạo mới lại mô hình (đảm bảo không bị dính trọng số cũ)
    model = MoEQFormerDeblurNet(
        in_channels=3,
        out_channels=3,
        num_feats=COMMON_CONFIG['num_feats'],
        num_blocks=COMMON_CONFIG['num_blocks'],
        num_queries=COMMON_CONFIG['num_queries'],
        num_experts=COMMON_CONFIG['num_experts']
    ).to(device)
    
    if cfg['freeze_backbone']:
        model.freeze_backbone(True)
        
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.AdamW(trainable_params, lr=cfg['lr'], weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg['epochs'], eta_min=1e-6)
    
    # 3. Vòng lặp train các epoch của cấu hình này
    history = {'train_loss': [], 'val_psnr': [], 'val_ssim': []}
    best_psnr = 0.0
    best_ssim = 0.0
    save_path = f"checkpoint_{exp_name}.pth"
    start_time = time.time()
    
    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        running_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"[{exp_name}] Epoch {epoch:02d}/{cfg['epochs']:02d}")
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            
            preds, aux_loss = model(inputs)
            recon_loss = criterion(preds, targets)
            total_loss = recon_loss + COMMON_CONFIG['aux_weight'] * aux_loss
            
            total_loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
            optimizer.step()
            
            running_loss += total_loss.item()
            pbar.set_postfix({'loss': f"{total_loss.item():.4f}"})
            
        epoch_loss = running_loss / len(train_loader)
        scheduler.step()
        
        # Đánh giá sau epoch
        val_psnr, val_ssim = evaluate_model(model, test_loader, max_eval_samples=COMMON_CONFIG['max_eval_samples'])
        history['train_loss'].append(epoch_loss)
        history['val_psnr'].append(val_psnr)
        history['val_ssim'].append(val_ssim)
        
        print(f"--> [{exp_name} | Epoch {epoch:02d}] Loss: {epoch_loss:.4f} | Test PSNR: {val_psnr:.2f} dB | Test SSIM: {val_ssim:.4f}")
        
        if val_psnr > best_psnr:
            best_psnr = val_psnr
            best_ssim = val_ssim
            torch.save(model.state_dict(), save_path)
            
    elapsed_time = (time.time() - start_time) / 60.0  # phút
    print(f" Hoàn thành {exp_name} trong {elapsed_time:.1f} phút. Best PSNR: {best_psnr:.2f} dB | Best SSIM: {best_ssim:.4f}")
    
    # Lưu kết quả tổng hợp của thí nghiệm này
    EXPERIMENT_RESULTS[exp_name] = {
        'config': cfg,
        'history': history,
        'best_psnr': best_psnr,
        'best_ssim': best_ssim,
        'final_loss': history['train_loss'][-1],
        'time_mins': elapsed_time,
        'checkpoint': save_path
    }

print("\n" + "="*65)
print(" ĐÃ HOÀN THÀNH TOÀN BỘ CÁC THÍ NGHIỆM TRONG CONFIG_LIST!")
print("="*65)

### Bảng Xếp Hạng So Sánh Kết Quả (Leaderboard)

In [ ]:
# Tạo bảng tổng hợp so sánh bằng Pandas DataFrame
summary_data = []
for name, res in EXPERIMENT_RESULTS.items():
    cfg = res['config']
    summary_data.append({
        'Tên Thí Nghiệm': name,
        'LR': cfg['lr'],
        'Batch Size': cfg['batch_size'],
        'Freeze': cfg['freeze_backbone'],
        'Epochs': cfg['epochs'],
        'Best PSNR (dB)': round(res['best_psnr'], 2),
        'Best SSIM': round(res['best_ssim'], 4),
        'Final Loss': round(res['final_loss'], 4),
        'Thời Gian (phút)': round(res['time_mins'], 1),
        'Mô Tả': cfg['description']
    })

df_summary = pd.DataFrame(summary_data)
# Sắp xếp theo PSNR cao nhất lên đầu
df_summary = df_summary.sort_values(by='Best PSNR (dB)', ascending=False).reset_index(drop=True)
df_summary.index += 1
df_summary.index.name = 'Hạng'

print(" BẢNG SO SÁNH HIỆU NĂNG CÁC CẤU HÌNH:")
display(df_summary)

best_exp_name = df_summary.iloc[0]['Tên Thí Nghiệm']
print(f"\n CẤU HÌNH TỐT NHẤT LÀ: [{best_exp_name}] với PSNR = {df_summary.iloc[0]['Best PSNR (dB)']} dB!")

### Biểu Đồ So Sánh Tiến Trình Hội Tụ Của Các Cấu Hình (Multi-Run Curves)

In [ ]:
# Vẽ các đường so sánh đè lên nhau trên cùng 1 biểu đồ
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# 1. So sánh Loss
for i, (name, res) in enumerate(EXPERIMENT_RESULTS.items()):
    c = colors[i % len(colors)]
    axes[0].plot(range(1, len(res['history']['train_loss']) + 1), res['history']['train_loss'], 
                 label=name, color=c, marker='o', linewidth=2)
axes[0].set_title('So Sánh Training Loss (Càng thấp càng tốt)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, linestyle='--', alpha=0.6)
axes[0].legend()

# 2. So sánh PSNR
for i, (name, res) in enumerate(EXPERIMENT_RESULTS.items()):
    c = colors[i % len(colors)]
    axes[1].plot(range(1, len(res['history']['val_psnr']) + 1), res['history']['val_psnr'], 
                 label=name, color=c, marker='s', linewidth=2)
axes[1].set_title('So Sánh Validation PSNR (Càng cao càng tốt)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('PSNR (dB)')
axes[1].grid(True, linestyle='--', alpha=0.6)
axes[1].legend()

# 3. So sánh SSIM
for i, (name, res) in enumerate(EXPERIMENT_RESULTS.items()):
    c = colors[i % len(colors)]
    axes[2].plot(range(1, len(res['history']['val_ssim']) + 1), res['history']['val_ssim'], 
                 label=name, color=c, marker='^', linewidth=2)
axes[2].set_title('So Sánh Validation SSIM (Càng gần 1.0 càng tốt)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('SSIM')
axes[2].grid(True, linestyle='--', alpha=0.6)
axes[2].legend()

plt.tight_layout()
plt.show()

### Trực Quan Hóa Kết Quả Của Mô Hình Tốt Nhất (Best Model Demo)
Tải trọng số của cấu hình đạt điểm cao nhất và hiển thị ảnh thực tế:

In [ ]:
# Tải checkpoint của Best Model
best_checkpoint_path = EXPERIMENT_RESULTS[best_exp_name]['checkpoint']
print(f" Đang tải mô hình tốt nhất từ: {best_checkpoint_path}")

best_model = MoEQFormerDeblurNet(
    in_channels=3, out_channels=3,
    num_feats=COMMON_CONFIG['num_feats'],
    num_blocks=COMMON_CONFIG['num_blocks'],
    num_queries=COMMON_CONFIG['num_queries'],
    num_experts=COMMON_CONFIG['num_experts']
).to(device)

best_model.load_state_dict(torch.load(best_checkpoint_path, map_location=device))
best_model.eval()

# Chọn ngẫu nhiên 3 ảnh từ tập GoPro test
indices = random.sample(range(len(test_ds)), min(3, len(test_ds)))

for idx in indices:
    blur_tensor, sharp_tensor = test_ds[idx]
    blur_input = blur_tensor.unsqueeze(0).to(device)
    
    with torch.no_grad():
        deblur_tensor, _ = best_model(blur_input)
        deblur_tensor = torch.clamp(deblur_tensor, 0.0, 1.0).squeeze(0).cpu()
        
    score_psnr = calc_psnr(deblur_tensor, sharp_tensor)
    score_ssim = calc_ssim(deblur_tensor, sharp_tensor)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    axes[0].imshow(blur_tensor.permute(1, 2, 0).numpy())
    axes[0].set_title("Input: Ảnh Mờ (Blur)", fontsize=12)
    axes[0].axis("off")
    
    axes[1].imshow(deblur_tensor.permute(1, 2, 0).numpy())
    axes[1].set_title(f"Output: {best_exp_name}\nPSNR: {score_psnr:.2f} dB | SSIM: {score_ssim:.4f}", 
                      fontsize=12, color='darkgreen', fontweight='bold')
    axes[1].axis("off")
    
    axes[2].imshow(sharp_tensor.permute(1, 2, 0).numpy())
    axes[2].set_title("Target: Ảnh Gốc Nét (Sharp)", fontsize=12)
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()